# Model Training Procedure

This notebook includes all parameters for model training and the selection process for further calssification in the active learning process.

---

## Install and Import Dependencies

In [8]:
!pip install iterative-stratification
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score, hamming_loss, classification_report
from datasets import Dataset, DatasetDict, load_dataset
import numpy as np
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from tqdm.auto import tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import Coded Segments

Segments were exported from MaxQDA after each round in a long format and appended to the previous round of training data. (e.g 400 --> 500 --> ...)

In [ ]:
df = pd.read_excel("/content/drive/MyDrive/Bosnian Parliment/700_prelim_coded.xlsx")

df = df[['Segment', 'Code']] # Eliminate extra columns

# Pivot the table into a binary matrix
# This creates a column for each of the 17 codes
multi_label_df = pd.get_dummies(df, columns=['Code'], prefix='', prefix_sep='')

# Group by the text segment to merge duplicates
# If a segment had 3 codes, it will now be one row with three classifications
multi_label_df = multi_label_df.groupby('Segment').max().reset_index()

# Create a 'labels' list column (preferred by Hugging Face)
cols = multi_label_df.columns[1:] # All columns except 'Segment'
multi_label_df['label_vector'] = multi_label_df[cols].values.tolist()

In [ ]:
# X = text segments, y = a matrix of 17 binary labels
X = multi_label_df['Segment'].values
y = np.array(multi_label_df['label_vector'].tolist())

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42) # Initialize the splitter
# This splitter was used to ensure "rare" classes are not omitted from the validation set

train_index, val_index = next(msss.split(X, y))

train_texts, val_texts = X[train_index].tolist(), X[val_index].tolist()
train_labels, val_labels = y[train_index].tolist(), y[val_index].tolist()

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

## Initialize Model and Create Dataset Object

Initial training runs ran model versions saved in the Google Drive. This example uses the current HuggingFace iteration for brevity.

In [ ]:
model_id = "zastuck/roberta-base-bosnian-parliament-multilabel-v1"
tokenizer = AutoTokenizer.from_pretrained(model_id) # Initialize the tokenizer

# Map the 17 codes
id2label = {i: label for i, label in enumerate(cols)}
label2id = {label: i for i, label in enumerate(cols)}
# These are unecessary after the first run, but do not hurt anything. The names are saved in the model config

model = AutoModelForSequenceClassification.from_pretrained( # Initialize model w/ parameters
    model_id,
    num_labels=17,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification" # CRITICAL for multi-label
)

In [ ]:
#Converting DF to Dataset


def preprocess_function(examples):
    # Standard RoBERTa tokenization
    encoding = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    # Binary labels must be floats for the BCE loss function
    encoding["labels"] = [list(map(float, l)) for l in examples["label"]]
    return encoding

# Create datasets
train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_ds = Dataset.from_dict({"text": val_texts, "label": val_labels})

# Map the tokenization
train_dataset = train_ds.map(preprocess_function, batched=True)
val_dataset = val_ds.map(preprocess_function, batched=True)

## Train the Model

This was done over 7 iterations in total, and used a few different settings. For brevity, only the initial and last setting are shown as examples

In [ ]:
# Create the metrics function used by the trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Use sigmoid to get probabilities, then threshold at 0.5
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > 0.5).astype(float)

    return {
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "f1_micro": f1_score(labels, predictions, average="micro"),
        "hamming_loss": hamming_loss(labels, predictions)
    }

### Initial Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       # Check performance after every pass
    save_strategy="epoch",
    learning_rate=2e-5,          # Higher learning rate for initial training
    per_device_train_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"    # Focus on F1, not accuracy
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

### Final Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       # Check performance after every pass
    save_strategy="epoch",
    learning_rate=5e-6,          # Lower learning rate for fine-tuning
    per_device_train_batch_size=7,
    label_smoothing_factor=0.1, # Keeps the model more "Open Minded" to deal with ambiguity
    num_train_epochs=10,         # This can be changed up to 15-20 in intermediate rounds
    weight_decay=0.3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"    # Focus on F1, not accuracy
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
output_path = "/content/drive/MyDrive/Bosnian Parliment/roberta-active-learning-v6"
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)

### Detailed Metrics

In [ ]:
def compute_metrics_all(eval_pred):
    # Unpack logits and labels
    logits, labels = eval_pred

    # Convert logits to probabilities (Sigmoid)
    probs = 1 / (1 + np.exp(-logits))

    # Threshold at 0.5 for hard 0/1 predictions
    predictions = (probs > 0.5).astype(int)

    # Generate the report using generic label indices (0 to 16)
    num_labels = labels.shape[1]
    generic_names = [f"Label_{i}" for i in range(num_labels)]

    report = classification_report(
        labels,
        predictions,
        target_names=generic_names,
        digits=4
    )

    # Print the report to the console/cell
    print("\n" + "="*60)
    print("DETAILED PER-LABEL PERFORMANCE")
    print("="*60)
    print(report)
    print("="*60 + "\n")

    # Return the metrics the Trainer expects to track
    return {
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'f1_micro': f1_score(labels, predictions, average='micro'),
        'hamming_loss': (labels != predictions).mean()
    }

trainer.compute_metrics = compute_metrics_all # Update the function
trainer.evaluate()

## Preparing Next Manual Coding Set

After a training round, all data was classified using the updated model and another round of 50-100 segments were exported to code in MaxQDA. These segments were chosed based on uncertainty of the model and the potential for "rare" classes.

In [5]:
data_path = "/content/drive/MyDrive/Bosnian Parliment/segmented_BiHCorp.parquet"
raw_dataset = load_dataset('parquet', data_files=data_path, split='train') # Train needs to be used even though there is no split

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
# Load the Model & Data
model_id = "zastuck/roberta-base-bosnian-parliament-multilabel-v1" # Load model from HuggingFace or change to local version
classifier = pipeline("text-classification", model=model_id, device=0, batch_size=32, top_k=None)

#  Generate Raw Probabilities
raw_scores = []
for out in tqdm(classifier(KeyDataset(raw_dataset, "segment")), total=len(raw_dataset)):
    # 'out' is a list: [{'label': 'PROCEDURAL', 'score': 0.9}, {'label': 'COLLECT_MEM', 'score': 0.1}, ...]
    # turns this into a dictionary: {'PROCEDURAL': 0.9, 'COLLECT_MEM': 0.1, ...}
    label_dict = {item['label']: item['score'] for item in out}
    raw_scores.append(label_dict)

# Creating the DataFrame from a list of dicts
df_results = pd.DataFrame(raw_scores)

# Add the original text and Id
df_results['segment'] = raw_dataset['segment']
df_results['segment_id'] = raw_dataset['segment_id']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  0%|          | 0/120783 [00:00<?, ?it/s]

,PROCEDURAL,SUBSTANTIVE_TECHNICAL,SUBSTANTIVE_IDENTITY,OTHER_INTERNAL,SELF_ETHNIC,ROUTINE_DEF,SELF_STATE,COLLECT_MEM,OTHER_ABSTRACT,SELF_PARTY,HIST_CONT,SELF_GOV,COND_CORRUPT,MORAL_AGENCY,SELF_TRANS,OTHER_EXTERNAL,EXIST_IDENTITY,segment,segment_id
0,0.960673,0.023110,0.018140,0.011891,0.011431,0.011158,0.010836,0.010040,0.009791,0.009700,0.009176,0.008935,0.007274,0.007190,0.007041,0.006908,0.006601,1. Answers to delegational questions and deleg...,1
1,0.898633,0.067123,0.012851,0.007996,0.008522,0.008063,0.008738,0.008024,0.007187,0.006860,0.007111,0.006449,0.005776,0.005670,0.005322,0.005499,0.004768,. Proposal for a Law on Amendments and Supplem...,2
2,0.740025,0.194571,0.010106,0.006149,0.006788,0.006427,0.007453,0.006406,0.005797,0.005393,0.006486,0.005149,0.004972,0.004660,0.004484,0.005185,0.004240,. Proposal for a law on the protection of new ...,3
3,0.933069,0.040504,0.014905,0.009156,0.009495,0.009236,0.009911,0.008654,0.007987,0.007759,0.007658,0.007285,0.006293,0.006480,0.005883,0.006072,0.005185,. Proposal for a Law on the Constitutional Cou...,4
4,0.885026,0.075180,0.012749,0.006730,0.007790,0.007840,0.010937,0.007695,0.007145,0.006092,0.007609,0.005591,0.005871,0.005561,0.005206,0.005772,0.004391,. Consideration of the Annual Platform on Secu...,5


In [11]:
probs = df_results.iloc[:, :16].values # Create numpy array

entropy = -(probs * np.log2(probs + 1e-10) + (1 - probs) * np.log2(1 - probs + 1e-10)).mean(axis=1) # Use entropy formula to determine uncertainty from logits
df_results['uncertainty_score'] = entropy

#Rare Classes
rare_class_indices = [5] # Replace with current rare class indices
# Take the maximum probability assigned to ANY of the rare classes
df_results['rare_potential'] = np.max(probs[:, rare_class_indices], axis=1)

# Selecting 70% Entropy based, 30% Rare classes (This changes to focus more on rare cases in later rounds)

# Get the top uncertain rows
top_uncertain = df_results.sort_values('uncertainty_score', ascending=False).head(70)

# Get the top rare potential rows (excluding those already in top_uncertain)
remaining = df_results.drop(top_uncertain.index)
top_rare = remaining.sort_values('rare_potential', ascending=False).head(30)

# Combine for the next MAXQDA batch
next_batch = pd.concat([top_uncertain, top_rare])
next_batch = next_batch[["segment_id", "segment", "uncertainty_score", "rare_potential"]]
next_batch.to_excel("/content/drive/MyDrive/Bosnian Parliment/next_batch.xlsx", index=False)
next_batch.head()

# Code this round manually in MaxQDA

,segment_id,segment,uncertainty_score,rare_potential
24997,24998,"... also, this law will not and cannot be the ...",0.450247,0.999273
30525,30526,". Why? I repeat, they were unconstitutional. N...",0.470085,0.999096
60682,60683,. I want to say that on such issues and on thi...,0.440151,0.999005
90011,90012,. Don't live in error and don't try to fool us...,0.474143,0.998858
90734,90735,". In the BiH Constitution, it has a suppositio...",0.373719,0.998831
